# Nexus Pipeline: Kokoro TTS + WhisperX (Auto-Sync)
Este notebook lê os arquivos `.txt` do seu Google Drive, gera o áudio com Kokoro (TTS SOTA) e executa o WhisperX para gerar os timestamps word-level (`.json`), salvando tudo de volta no Drive.

In [ ]:
# @title 1. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# @title 2. Instalar Dependências (Kokoro + WhisperX)
!apt-get update && apt-get install -y espeak-ng

# 1. Instalar dependências base com versões compatíveis
!pip install -q "pandas<3.0.0" "numpy<2.0.0" "huggingface-hub<1.0.0" soundfile

# 2. Instalar Kokoro
!pip install -q kokoro

# 3. Instalar WhisperX sem reinstalar dependências conflitantes
!pip install -q git+https://github.com/m-bain/whisperx.git --no-deps
!pip install -q faster-whisper nltk

# 4. Forçar versão correta de transformers para resolver o ImportError
!pip install -q -U transformers

print("✅ Dependências instaladas!")
print("⚠️  IMPORTANTE: Vá em 'Ambiente de Execução' -> 'Reiniciar sessão' antes de prosseguir.")

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
espeak-ng is already the newest version (1.50+dfsg-10ubuntu0.1).
0 upgraded, 0 newly installed

In [ ]:
# @title 3. Executar Pipeline (TTS -> Align -> Save)
import os
import torch
import soundfile as sf
import numpy as np
import json
import whisperx
from kokoro import KPipeline

# Configurações de Diretório no Drive
BASE_DIR = "/content/drive/MyDrive/nexus_pipeline"
TEXTS_DIR = os.path.join(BASE_DIR, "texts")
AUDIO_READY_DIR = os.path.join(BASE_DIR, "audio_ready")

# Criar pastas se não existirem
os.makedirs(TEXTS_DIR, exist_ok=True)
os.makedirs(AUDIO_READY_DIR, exist_ok=True)

if not os.listdir(TEXTS_DIR):
    print(f"⚠️  Atenção: Coloque seus arquivos .txt em {TEXTS_DIR}")
else:
    print("[Inicializando Modelos]")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    k_pipeline = KPipeline(lang_code='a')
    whisper_model = whisperx.load_model("base", device, compute_type="float16" if device == "cuda" else "int8")

    for filename in os.listdir(TEXTS_DIR):
        if not filename.endswith(".txt"): continue

        scene_id = filename.replace(".txt", "")
        txt_path = os.path.join(TEXTS_DIR, filename)
        wav_path = os.path.join(AUDIO_READY_DIR, f"{scene_id}.wav")
        json_path = os.path.join(AUDIO_READY_DIR, f"{scene_id}_words.json")

        if os.path.exists(wav_path) and os.path.exists(json_path):
            print(f"[Pulo] Cena {scene_id} já processada.")
            continue

        print(f"\n--- Processando: {scene_id} ---")
        with open(txt_path, 'r', encoding='utf-8') as f: text = f.read().strip()
        if not text: continue

        # 1. Kokoro TTS
        generator = k_pipeline(text, voice='af_heart', speed=1)
        all_audio = [audio for _, _, audio in generator if audio is not None]
        if all_audio:
            final_audio = np.concatenate(all_audio)
            sf.write(wav_path, final_audio, 24000)

        # 2. WhisperX Alignment
        audio_data = whisperx.load_audio(wav_path)
        result = whisper_model.transcribe(audio_data, batch_size=16)
        model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
        result = whisperx.align(result["segments"], model_a, metadata, audio_data, device)

        words_list = []
        for segment in result["segments"]:
            for word in segment.get("words", []):
                if "start" in word:
                    words_list.append({"word": word["word"], "start": word["start"], "end": word["end"]})

        with open(json_path, 'w', encoding='utf-8') as f: json.dump(words_list, f, indent=2)
        print(f"✅ Cena {scene_id} concluída.")

[Inicializando Modelos]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/2.35k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


kokoro-v1_0.pth:   0%|          | 0.00/327M [00:00<?, ?B/s]

...


--- Processando: scene_03_the_cost ---


voices/af_heart.pt:   0%|          | 0.00/523k [00:00<?, ?B/s]

2026-06-04 22:47:45 - whisperx.asr - WARNING - Audio is shorter than 30s, language detection may be inaccurate
2026-06-04 22:47:48 - whisperx.asr - INFO - Detected language: en (1.00) in first 30s of audio
Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth" to /root/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960_asr_ls960.pth


100%|██████████| 360M/360M [00:01<00:00, 283MB/s]


✅ Cena scene_03_the_cost concluída.

--- Processando: scene_01_cold_war_start ---
2026-06-04 22:48:22 - whisperx.asr - WARNING - Audio is shorter than 30s, language detection may be inaccurate
2026-06-04 22:48:24 - whisperx.asr - INFO - Detected language: en (1.00) in first 30s of audio
✅ Cena scene_01_cold_war_start concluída.

--- Processando: scene_02_military_spending ---
2026-06-04 22:48:53 - whisperx.asr - WARNING - Audio is shorter than 30s, language detection may be inaccurate
2026-06-04 22:48:56 - whisperx.asr - INFO - Detected language: en (1.00) in first 30s of audio
✅ Cena scene_02_military_spending concluída.
